# 🇵🇱 Dane Ekonomiczne Polski

Notebook zbiera dane z trzech źródeł:

| # | Dane | Źródło | Metoda |
|---|------|--------|--------|
| 1 | **Inflacja (CPI)** | Stooq.pl | Web scraping |
| 2 | **Stopy procentowe NBP** | NBP OpenData API | REST API |
| 3 | **Dane lokalne** (bezrobocie, zarobki, ludność, gęstość) | Bank Danych Lokalnych GUS | REST API |

> **Wymagania:** `requests`, `pandas`, `beautifulsoup4`, `lxml`  
> Zainstaluj: `pip install requests pandas beautifulsoup4 lxml`

---
## 0. Importy i konfiguracja

In [ ]:
import requests
import pandas as pd
import time
import json
from bs4 import BeautifulSoup
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Wspólne nagłówki HTTP ──────────────────────────────────────────────────────
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'pl-PL,pl;q=0.9,en;q=0.8',
}

print(f"✅ Biblioteki załadowane | {datetime.now().strftime('%Y-%m-%d %H:%M')}")

---
## 1. 📈 Inflacja – scraping Stooq.pl

Stooq udostępnia szeregi makroekonomiczne w formacie CSV przez prosty URL (bez klucza API).  
Używamy tickerów:
- `CPIYPL.M` → CPI r/r miesięczny (rok do roku)
- `CPIMPL.M` → CPI m/m miesięczny (miesiąc do miesiąca)

In [ ]:
def pobierz_cpi_stooq(ticker: str, nazwa: str) -> pd.DataFrame:
    """
    Pobiera dane CPI z Stooq.pl (format CSV przez URL).
    
    Parameters
    ----------
    ticker : str  – symbol szeregu na Stooq, np. 'CPIYPL.M'
    nazwa  : str  – przyjazna nazwa kolumny wynikowej
    
    Returns
    -------
    pd.DataFrame z kolumnami [Date, <nazwa>]
    """
    url = f'https://stooq.pl/q/d/l/?s={ticker}&i=m'
    print(f'  → Pobieranie: {url}')
    
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    
    from io import StringIO
    df = pd.read_csv(StringIO(resp.text))
    
    # Stooq zwraca kolumny: Date, Open, High, Low, Close, Volume
    # Wartość wskaźnika jest w 'Close'
    df = df[['Date', 'Close']].rename(columns={'Close': nazwa})
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    
    print(f'  ✅ {nazwa}: {len(df)} rekordów '
          f'({df["Date"].min().date()} – {df["Date"].max().date()})')
    return df


print('📥 Pobieranie inflacji z Stooq.pl...')
cpi_yoy = pobierz_cpi_stooq('CPIYPL.M', 'CPI_rdr_%')
time.sleep(1)  # grzeczna pauza między żądaniami
cpi_mom = pobierz_cpi_stooq('CPIMPL.M', 'CPI_mdm_%')

In [ ]:
# Połącz oba szeregi w jedną tabelę
inflacja = pd.merge(cpi_yoy, cpi_mom, on='Date', how='outer').sort_values('Date')
inflacja['Rok']     = inflacja['Date'].dt.year
inflacja['Miesiac'] = inflacja['Date'].dt.month

print(f'\n📊 Inflacja – ostatnie 12 miesięcy:')
inflacja.tail(12)[['Date','CPI_rdr_%','CPI_mdm_%']]

In [ ]:
# Zapis do pliku
inflacja.to_csv('inflacja_polska.csv', index=False, encoding='utf-8-sig')
print('💾 Zapisano → inflacja_polska.csv')
inflacja.describe()

---
## 2. 🏦 Stopy procentowe NBP – oficjalne API

Narodowy Bank Polski udostępnia dane przez **api.nbp.pl** w formacie JSON.  
Używamy tabeli stóp procentowych NBP:  
`https://api.nbp.pl/api/cenyczynnikow/stopyprocentowe/`

Dostępne stopy: referencyjna, lombardowa, depozytowa, redyskontowa weksli.

In [ ]:
def pobierz_stopy_nbp() -> pd.DataFrame:
    """
    Pobiera historyczne stopy procentowe NBP przez oficjalne API.
    Endpoint: https://api.nbp.pl/api/cenyczynnikow/stopyprocentowe/
    """
    url = 'https://api.nbp.pl/api/cenyczynnikow/stopyprocentowe/'
    print(f'  → Zapytanie: {url}')
    
    resp = requests.get(
        url,
        headers={**HEADERS, 'Accept': 'application/json'},
        timeout=15
    )
    resp.raise_for_status()
    data = resp.json()
    
    rekordy = []
    for wpis in data:
        data_obowiazywania = wpis.get('obowiazujeOd')
        for stopa in wpis.get('stopy', []):
            rekordy.append({
                'Data':         data_obowiazywania,
                'Rodzaj_stopy': stopa.get('rodzaj', '').strip(),
                'Wartosc_%':    stopa.get('oprocentowanie')
            })
    
    df = pd.DataFrame(rekordy)
    df['Data'] = pd.to_datetime(df['Data'])
    df = df.sort_values(['Rodzaj_stopy', 'Data']).reset_index(drop=True)
    
    print(f'  ✅ Pobrano {len(data)} zestawów decyzji, '
          f'{len(df)} wierszy (wszystkie stopy)')
    return df


print('📥 Pobieranie stóp procentowych NBP...')
stopy = pobierz_stopy_nbp()

In [ ]:
# Przekształć do formatu szerokiego (każda stopa = kolumna)
stopy_wide = stopy.pivot_table(
    index='Data',
    columns='Rodzaj_stopy',
    values='Wartosc_%'
).reset_index()
stopy_wide.columns.name = None

print('📊 Stopy procentowe NBP – ostatnie 10 decyzji:')
stopy_wide.tail(10)

In [ ]:
# Zapis do pliku
stopy_wide.to_csv('stopy_procentowe_nbp.csv', index=False, encoding='utf-8-sig')
print('💾 Zapisano → stopy_procentowe_nbp.csv')

# Podgląd dostępnych typów stóp
print('\nDostępne rodzaje stóp:')
print(stopy['Rodzaj_stopy'].unique())

---
## 3. 🗺️ Bank Danych Lokalnych GUS – API BDL

**API:** `https://bdl.stat.gov.pl/api/v1/`  

Pobieramy cztery grupy wskaźników dla **wszystkich powiatów** (unit-level=5):

| Wskaźnik | ID zmiennej BDL | Ziarnistość |
|----------|----------------|-------------|
| Stopa bezrobocia rejestrowanego | `60559` | miesięcznie |
| Przeciętne wynagrodzenie brutto | `64428` | rocznie |
| Liczba ludności | `72305` | rocznie |
| Gęstość zaludnienia (os./km²) | `60107` | rocznie |

> **Klucz API:** opcjonalny, bez klucza limit to ~100 req/h. Ustaw `BDL_API_KEY` jeśli masz.

In [ ]:
# ── Konfiguracja BDL ────────────────────────────────────────────────────────────
BDL_BASE   = 'https://bdl.stat.gov.pl/api/v1'
BDL_API_KEY = ''   # ← wstaw swój klucz z https://api.stat.gov.pl/Home/BdlApi
                   #   lub pozostaw pusty (bez klucza działa, ale z limitami)

# Identyfikatory zmiennych BDL
ZMIENNE_BDL = {
    'stopa_bezrobocia_%':       60559,  # Stopa bezrobocia rejestrowanego (miesięczna)
    'przec_wynagrodzenie_brutto': 64428, # Przeciętne wynagrodzenie brutto (roczna)
    'liczba_ludnosci':           72305,  # Ludność ogółem (roczna)
    'gestosc_zaludnienia':       60107,  # Gęstość zaludnienia na km² (roczna)
}

# Poziomy agregacji: 5 = powiat, 6 = gmina
UNIT_LEVEL = 5  # zmień na 6 dla gmin (dużo więcej rekordów!)

def bdl_headers() -> dict:
    h = {**HEADERS, 'Accept': 'application/json'}
    if BDL_API_KEY:
        h['X-ClientId'] = BDL_API_KEY
    return h

print(f'⚙️  Konfiguracja BDL: unit-level={UNIT_LEVEL} '
      f'({"powiat" if UNIT_LEVEL==5 else "gmina"})'
      f', klucz API: {"ustawiony" if BDL_API_KEY else "brak (tryb anonimowy)"}')

In [ ]:
def bdl_pobierz_zmienna(
    var_id: int,
    nazwa_kolumny: str,
    unit_level: int = 5,
    page_size: int = 100
) -> pd.DataFrame:
    """
    Pobiera wszystkie dane dla jednej zmiennej BDL, stronicując przez wyniki.
    
    Parameters
    ----------
    var_id        : ID zmiennej BDL
    nazwa_kolumny : nazwa kolumny w wynikowym DataFrame
    unit_level    : 5=powiat, 6=gmina
    page_size     : liczba rekordów na stronę (max 100)
    
    Returns
    -------
    pd.DataFrame z kolumnami [unit_id, unit_name, year, period, <nazwa_kolumny>]
    """
    rekordy = []
    strona = 0
    
    while True:
        params = {
            'unit-level': unit_level,
            'page':       strona,
            'page-size':  page_size,
            'format':     'json',
        }
        url = f'{BDL_BASE}/data/by-variable/{var_id}'
        
        try:
            resp = requests.get(url, params=params,
                                headers=bdl_headers(), timeout=30)
            resp.raise_for_status()
        except requests.HTTPError as e:
            print(f'    ⚠️  HTTP {resp.status_code} na stronie {strona}: {e}')
            break
        
        payload = resp.json()
        wyniki  = payload.get('results', [])
        
        if not wyniki:
            break
        
        for jednostka in wyniki:
            uid   = jednostka.get('id')
            uname = jednostka.get('name')
            for obs in jednostka.get('values', []):
                rekordy.append({
                    'unit_id':      uid,
                    'unit_name':    uname,
                    'year':         obs.get('year'),
                    'period':       obs.get('period'),  # numer okresu (miesiąc)
                    nazwa_kolumny:  obs.get('val'),
                })
        
        # Sprawdź czy jest kolejna strona
        links = payload.get('links', {})
        if not links.get('next'):
            break
        
        strona += 1
        time.sleep(0.3)  # grzeczna pauza
        
        if strona % 10 == 0:
            print(f'    ... strona {strona}, pobrano {len(rekordy):,} rekordów')
    
    df = pd.DataFrame(rekordy)
    print(f'  ✅ {nazwa_kolumny}: {len(df):,} rekordów '
          f'({df["unit_id"].nunique()} jednostek)')
    return df


print('Funkcja bdl_pobierz_zmienna() gotowa.')

### 3a. Stopa bezrobocia rejestrowanego (miesięczna)

In [ ]:
print('📥 Pobieranie stopy bezrobocia z BDL (może chwilę potrwać)...')
df_bezrobocie = bdl_pobierz_zmienna(
    var_id=ZMIENNE_BDL['stopa_bezrobocia_%'],
    nazwa_kolumny='stopa_bezrobocia_%',
    unit_level=UNIT_LEVEL
)

# Dodaj kolumnę z datą (rok + miesiąc)
df_bezrobocie['data'] = pd.to_datetime(
    df_bezrobocie['year'].astype(str) + '-' +
    df_bezrobocie['period'].fillna(1).astype(int).astype(str).str.zfill(2),
    format='%Y-%m', errors='coerce'
)

print('\n📊 Podgląd – 5 losowych powiatów, ostatni dostępny miesiąc:')
ostatni = df_bezrobocie.dropna(subset=['stopa_bezrobocia_%'])
ostatni = ostatni[ostatni['data'] == ostatni['data'].max()]
ostatni.sample(min(5, len(ostatni)))[['unit_name','data','stopa_bezrobocia_%']]

### 3b. Przeciętne wynagrodzenie brutto (roczne)

In [ ]:
print('📥 Pobieranie przeciętnych wynagrodzeń z BDL...')
df_wynagrodzenia = bdl_pobierz_zmienna(
    var_id=ZMIENNE_BDL['przec_wynagrodzenie_brutto'],
    nazwa_kolumny='przec_wynagr_brutto_zl',
    unit_level=UNIT_LEVEL
)

print('\n📊 Podgląd – wynagrodzenia w ostatnim roku:')
ostatni_rok = df_wynagrodzenia.dropna(subset=['przec_wynagr_brutto_zl'])
ostatni_rok = ostatni_rok[ostatni_rok['year'] == ostatni_rok['year'].max()]
(
    ostatni_rok[['unit_name','year','przec_wynagr_brutto_zl']]
    .sort_values('przec_wynagr_brutto_zl', ascending=False)
    .head(10)
)

### 3c. Liczba ludności (roczna)

In [ ]:
print('📥 Pobieranie liczby ludności z BDL...')
df_ludnosc = bdl_pobierz_zmienna(
    var_id=ZMIENNE_BDL['liczba_ludnosci'],
    nazwa_kolumny='liczba_ludnosci',
    unit_level=UNIT_LEVEL
)

print('\n📊 Podgląd – 10 największych powiatów wg liczby ludności:')
ostatni_rok_lud = df_ludnosc.dropna(subset=['liczba_ludnosci'])
ostatni_rok_lud = ostatni_rok_lud[ostatni_rok_lud['year'] == ostatni_rok_lud['year'].max()]
(
    ostatni_rok_lud[['unit_name','year','liczba_ludnosci']]
    .sort_values('liczba_ludnosci', ascending=False)
    .head(10)
)

### 3d. Gęstość zaludnienia (roczna)

In [ ]:
print('📥 Pobieranie gęstości zaludnienia z BDL...')
df_gestosc = bdl_pobierz_zmienna(
    var_id=ZMIENNE_BDL['gestosc_zaludnienia'],
    nazwa_kolumny='gestosc_zaludnienia_os_km2',
    unit_level=UNIT_LEVEL
)

print('\n📊 Podgląd – 10 najgęściej zaludnionych powiatów:')
ostatni_rok_gest = df_gestosc.dropna(subset=['gestosc_zaludnienia_os_km2'])
ostatni_rok_gest = ostatni_rok_gest[ostatni_rok_gest['year'] == ostatni_rok_gest['year'].max()]
(
    ostatni_rok_gest[['unit_name','year','gestosc_zaludnienia_os_km2']]
    .sort_values('gestosc_zaludnienia_os_km2', ascending=False)
    .head(10)
)

---
## 4. 🔗 Połączenie danych BDL

Łączymy wszystkie dane BDL w jeden DataFrame wg `unit_id` i `year`.

In [ ]:
# Dane roczne: wynagrodzenie, ludność, gęstość
# (bezrobocie zostawiamy osobno, bo jest miesięczne)

ROCZNE = [
    (df_wynagrodzenia, 'przec_wynagr_brutto_zl'),
    (df_ludnosc,       'liczba_ludnosci'),
    (df_gestosc,       'gestosc_zaludnienia_os_km2'),
]

# Baza: wynagrodzenia
bdl_roczne = df_wynagrodzenia[['unit_id','unit_name','year','przec_wynagr_brutto_zl']].copy()

for df_src, kol in ROCZNE[1:]:
    tmp = df_src[['unit_id','year', kol]].copy()
    bdl_roczne = bdl_roczne.merge(tmp, on=['unit_id','year'], how='outer')

# unit_name mogło spaść przy outer merge – uzupełnij
bdl_roczne['unit_name'] = bdl_roczne.groupby('unit_id')['unit_name'].transform(
    lambda s: s.ffill().bfill()
)

print(f'📊 Połączone dane roczne BDL: {bdl_roczne.shape}')
print(f'   Jednostek: {bdl_roczne["unit_id"].nunique()}')
print(f'   Lata: {sorted(bdl_roczne["year"].dropna().unique().astype(int))}')
bdl_roczne.head()

In [ ]:
# Zapis danych
bdl_roczne.to_csv('bdl_powiaty_roczne.csv', index=False, encoding='utf-8-sig')
df_bezrobocie.to_csv('bdl_powiaty_bezrobocie_miesieczne.csv', index=False, encoding='utf-8-sig')

print('💾 Zapisano:')
print('   → bdl_powiaty_roczne.csv          (wynagrodzenia, ludność, gęstość)')
print('   → bdl_powiaty_bezrobocie_miesieczne.csv  (stopa bezrobocia, co miesiąc)')

---
## 5. 📋 Podsumowanie zebranych danych

In [ ]:
podsumowanie = {
    'Inflacja CPI (Stooq)': {
        'Rekordy':       len(inflacja),
        'Zakres dat':    f"{inflacja['Date'].min().date()} – {inflacja['Date'].max().date()}",
        'Plik':          'inflacja_polska.csv',
    },
    'Stopy procentowe NBP': {
        'Rekordy':       len(stopy_wide),
        'Zakres dat':    f"{stopy_wide['Data'].min().date()} – {stopy_wide['Data'].max().date()}",
        'Plik':          'stopy_procentowe_nbp.csv',
    },
    'BDL – dane roczne (powiaty)': {
        'Rekordy':       len(bdl_roczne),
        'Zakres lat':    f"{int(bdl_roczne['year'].min())} – {int(bdl_roczne['year'].max())}",
        'Plik':          'bdl_powiaty_roczne.csv',
    },
    'BDL – bezrobocie miesięczne': {
        'Rekordy':       len(df_bezrobocie),
        'Zakres dat':    f"{df_bezrobocie['data'].min().date()} – {df_bezrobocie['data'].max().date()}",
        'Plik':          'bdl_powiaty_bezrobocie_miesieczne.csv',
    },
}

print('=' * 65)
print('  PODSUMOWANIE ZEBRANYCH DANYCH')
print('=' * 65)
for nazwa, info in podsumowanie.items():
    print(f'\n📦 {nazwa}')
    for k, v in info.items():
        print(f'   {k:<15}: {v}')
print('\n' + '=' * 65)

---
## 6. 🛠️ Narzędzia pomocnicze

### Wyszukiwarka zmiennych BDL

Jeśli chcesz znaleźć inne wskaźniki w BDL, użyj poniższej funkcji.

In [ ]:
def szukaj_zmiennych_bdl(fraza: str, max_wynikow: int = 20) -> pd.DataFrame:
    """
    Wyszukuje zmienne w BDL po frazie tekstowej.
    
    Przykład:
        szukaj_zmiennych_bdl('wynagrodzenie')
        szukaj_zmiennych_bdl('PKB')
    """
    url    = f'{BDL_BASE}/variables/search'
    params = {'name': fraza, 'page': 0, 'page-size': max_wynikow}
    
    resp = requests.get(url, params=params, headers=bdl_headers(), timeout=15)
    resp.raise_for_status()
    
    wyniki = resp.json().get('results', [])
    df = pd.DataFrame([{
        'var_id':       w['id'],
        'nazwa':        w.get('n1', ''),
        'jednostka':    w.get('measureUnitName', ''),
        'subject_id':   w.get('subjectId', ''),
    } for w in wyniki])
    
    print(f'Znaleziono {len(df)} zmiennych dla frazy "{fraza}"')
    return df


# Przykładowe wyszukiwania:
# szukaj_zmiennych_bdl('wynagrodzenie')
# szukaj_zmiennych_bdl('PKB')
# szukaj_zmiennych_bdl('bezrobocie')

print('ℹ️  Odkomentuj i uruchom dowolną linię powyżej, aby wyszukać zmienne BDL.')

In [ ]:
def pobierz_liste_powiatow(unit_level: int = 5) -> pd.DataFrame:
    """
    Pobiera listę wszystkich jednostek terytorialnych danego poziomu.
    unit_level: 2=województwo, 5=powiat, 6=gmina
    """
    url    = f'{BDL_BASE}/units'
    params = {'level': unit_level, 'page': 0, 'page-size': 500, 'format': 'json'}
    
    resp = requests.get(url, params=params, headers=bdl_headers(), timeout=30)
    resp.raise_for_status()
    
    wyniki = resp.json().get('results', [])
    df = pd.DataFrame([{
        'unit_id':    u['id'],
        'unit_name':  u['name'],
        'parent_id':  u.get('parentId'),
        'kind':       u.get('kind'),
    } for u in wyniki])
    
    print(f'Pobrano {len(df)} jednostek poziomu {unit_level}')
    return df


# Odkomentuj, aby zobaczyć listę wszystkich powiatów z ID:
# powiaty = pobierz_liste_powiatow(5)
# powiaty.head(20)

print('ℹ️  Odkomentuj powyżej, aby pobrać listę powiatów z identyfikatorami TERYT.')